# AI-Driven Tourism Recommendation System
## DTS114TC — Software Component

This notebook uses LLM to generate a complete tourism recommendation web application.
All code, diagrams, and documentation are automatically generated following the AI-Driven SDLC methodology.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))

# Check dependencies
try:
    from PIL import Image
    print("Pillow OK")
except Exception:
    print("Installing Pillow...")
    import subprocess; subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pillow'])

from utils import load_environment, get_completion, setup_llm_client
from utils import clean_llm_output, recommended_models_table, save_artifact, load_artifact
from utils import render_plantuml_diagram, get_image_generation_completion
from IPython.display import display, Markdown, Image as IPyImage

print("All imports loaded.")

In [ ]:
# Load environment and setup client
load_environment()

MODEL = "openai/gpt-5.2"
IMAGE_MODEL = "qwen/qwen-image-2512"

client, model_name, provider = setup_llm_client(MODEL)
print(f"Provider: {provider}, Model: {model_name}")

# Quick check available models
_ = recommended_models_table(task="text", min_context=100_000)

## Phase 1: Inception
Define business intent and generate SDLC documentation.

In [ ]:
# Business problem definition
business_problem = (
    "Our company needs a tourism recommendation platform where users can input a city name "
    "and get a curated list of famous attractions along with a personalized travel itinerary. "
    "The system should display AI-generated images of each attraction and export a day-by-day plan."
)

#### Generate Problem Statement

In [ ]:
# Generate a focused problem statement from the business problem
prompt = f"Turn this business problem into a single, clear problem statement (2-3 sentences max).
Business Problem: {business_problem}"
problem_statement = get_completion(prompt, client, model_name, provider, temperature=0.3)
print(problem_statement)

#### Generate User Personas

In [ ]:
# Generate personas based on the problem statement
prompt = (
    "Generate 4 user personas for a tourism recommendation platform. Use this format:
"
    "1. **Role Title**
   - Responsibilities: ...
   - Needs: ...

"
    "Rules: single role titles only, no combined titles (no slashes). 2-3 bullets per role.
"
    f"Problem Statement: {problem_statement}"
)
personas = get_completion(prompt, client, model_name, provider, temperature=0.3)
print(personas)

#### Generate Product Requirements Document

In [ ]:
# Generate PRD from problem statement and personas
prompt = (
    "Write a PRD in markdown with these headings, 2-4 concise bullets each:
"
    "## Overview
## Goals
## Non-Goals
## User Personas (brief)
"
    "## Key Features
## User Flows
## Functional Requirements
"
    "## Non-Functional Requirements
## Constraints/Assumptions
## Success Metrics

"
    "Rules: only the headings above, no extra sections. Keep bullets short.
"
    f"Problem Statement: {problem_statement}
"
    f"Personas: {personas}"
)
prd = get_completion(prompt, client, model_name, provider, temperature=0.3)
prd = clean_llm_output(prd, language='markdown')

os.makedirs('artifacts', exist_ok=True)
with open('artifacts/prd.md', 'w', encoding='utf-8') as f:
    f.write(prd)
print("Saved: artifacts/prd.md")
print(prd[:500])

#### Generate User Stories

In [ ]:
# Generate user stories as structured JSON
prompt = (
    "Return ONLY valid JSON following this schema exactly:
"
    '{
  "user_stories": [
    {
      "id": 1,
'
    '      "role": "<role>",
      "goal": "<goal>",
'
    '      "benefit": "<benefit>",
'
    '      "acceptance_criteria": ["<criteria>", "<criteria>"]
    }
  ]
}

'
    "Rules: 5 stories total. Keep each field concise. No extra keys. Role names must be singular.
"
    f"PRD: {prd}"
)
user_stories = get_completion(prompt, client, model_name, provider, temperature=0.3)
user_stories = clean_llm_output(user_stories, language='json')

import json
os.makedirs('artifacts', exist_ok=True)
with open('artifacts/user_stories.json', 'w', encoding='utf-8') as f:
    json.dump({'user_stories': user_stories}, f, indent=2, ensure_ascii=False)
print("Saved: artifacts/user_stories.json")
print(user_stories[:400])

## Phase 2: Construction
Generate system design (UML diagrams) and implementation code.

#### Generate UML Use Case Diagram

In [ ]:
# Generate PlantUML Use Case diagram from user stories
prompt = (
    "Generate a UML-compliant PlantUML use case diagram from these user stories.
"
    "Requirements:
"
    "- Define actors with the `actor` keyword, placed outside the system boundary
"
    "- Wrap use cases in `rectangle \"System\" {{ ... }}`
"
    "- Use `--` for actor-to-usecase associations (not arrows)
"
    "- One use case per distinct goal, named with verb-noun phrases (max 5 words)
"
    "- Extract actors from the `role` field, use singular names
"
    "- Return ONLY valid PlantUML code, no explanations

"
    f"User Stories: {user_stories}"
)
puml_uc = get_completion(prompt, client, model_name, provider, temperature=0.3)
puml_uc = clean_llm_output(puml_uc, language='text')
print(puml_uc)

os.makedirs('artifacts/diagrams', exist_ok=True)
with open('artifacts/diagrams/use_case.puml', 'w') as f:
    f.write(puml_uc)
print("Saved: artifacts/diagrams/use_case.puml")

# Render to PNG
render_plantuml_diagram(puml_uc, "artifacts/diagrams/use_case.png")

#### Generate UML Class Diagram

In [ ]:
# Generate a UML class diagram for the system architecture
prompt = (
    "Generate a PlantUML class diagram for a tourism recommendation system.
"
    "Include these entities: User, City, Attraction, TravelPlan, ItineraryItem.
"
    "Show relationships, attributes, and methods. Use standard UML notation.
"
    "Return ONLY valid PlantUML code, no explanations.
"
    f"PRD Summary: {prd[:300]}"
)
puml_class = get_completion(prompt, client, model_name, provider, temperature=0.3)
puml_class = clean_llm_output(puml_class, language='text')
print(puml_class)

with open('artifacts/diagrams/class_diagram.puml', 'w') as f:
    f.write(puml_class)
print("Saved: artifacts/diagrams/class_diagram.puml")

render_plantuml_diagram(puml_class, "artifacts/diagrams/class_diagram.png")